<a href="https://colab.research.google.com/github/PEDRO-AK47/Proyecto/blob/main/Evaluacion1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Learning
## Evaluación Parcial N° 1 - DLY0100

**Integrantes:**
* Daniel Azocar
* Martin Veliz
* Pedro Santibañez



## 1. Introducción

El presente proyecto tiene como objetivo resolver un problema de clasificación de imágenes utilizando los fundamentos de Deep Learning. Para ello, se implementará una Red Neuronal Artificial del tipo Perceptrón Multicapa (MLP).

El conjunto de datos seleccionado para este desafío es **Fashion-MNIST**, un dataset estandarizado que contiene 70.000 imágenes en escala de grises (28x28 píxeles) distribuidas en 10 categorías diferentes de prendas de vestir.

**Objetivos del Modelo:**
* Desarrollar una arquitectura MLP capaz de clasificar correctamente las prendas con alta precisión.
* Analizar el impacto de distintos hiperparámetros (épocas, tasa de aprendizaje, tamaño de *batch*).
* Comparar diferentes funciones de activación y error para encontrar la configuración óptima.
* Implementar técnicas de optimización y regularización para prevenir el sobreajuste (*overfitting*) y asegurar la estabilidad de la red.

## 2. Carga y Preprocesamiento de Datos

Para alimentar nuestra red neuronal, utilizaremos la API de Keras para descargar el dataset Fashion-MNIST. Este dataset ya viene dividido en un conjunto de entrenamiento (60.000 imágenes) y uno de prueba (10.000 imágenes).

**Justificación del preprocesamiento:**
Los valores de los píxeles en las imágenes originales van de 0 a 255. Si introducimos estos valores crudos a la red neuronal, los gradientes podrían desestabilizarse durante el entrenamiento, haciendo que la convergencia sea muy lenta o fallida.

Para solucionar esto, aplicamos una **normalización**: dividimos todos los valores entre 255.0. De esta forma, reescalamos las características para que estén en un rango de 0 a 1. Esto estabiliza matemáticamente el algoritmo de descenso del gradiente y permite que la red aprenda los pesos de manera más eficiente.

In [1]:
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import numpy as np

# 1. Carga de los datos
fashion_mnist = keras.datasets.fashion_mnist
(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()

# Nombres de las clases
class_names = ['Camiseta/Top', 'Pantalón', 'Suéter', 'Vestido', 'Abrigo',
               'Sandalia', 'Camisa', 'Zapatilla', 'Bolso', 'Botín']

# 2. Preprocesamiento: Normalización
train_images = train_images / 255.0
test_images = test_images / 255.0

# Verificamos la carga y el formato
print(f"Formato de entrenamiento: {train_images.shape} (Imágenes, Alto, Ancho)")
print(f"Cantidad de etiquetas de entrenamiento: {len(train_labels)}")
print(f"Formato de prueba: {test_images.shape}")

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Formato de entrenamiento: (60000, 28, 28) (Imágenes, Alto, Ancho)
Cantidad de etiquetas de entrenamiento: 60000
Formato de prueba: (10000, 28, 28)


## 3. Definición del Modelo Base (MLP) y Funciones

In [2]:
# 3. Construcción del modelo MLP base
# Definimos una arquitectura inicial con dos capas ocultas
modelo_base = keras.Sequential([
    keras.layers.Flatten(input_shape=(28, 28), name="Capa_Entrada"),
    keras.layers.Dense(128, activation='relu', name="Capa_Oculta_1"),
    keras.layers.Dense(64, activation='relu', name="Capa_Oculta_2"),
    keras.layers.Dense(10, activation='softmax', name="Capa_Salida")
])

# Visualizamos la estructura paramétrica de la red
print("Arquitectura del Modelo Base:")
modelo_base.summary()

# 4. Compilación del modelo
# Utilizamos el optimizador Adam por su eficiencia en ajustar la tasa de aprendizaje dinámicamente
modelo_base.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Arquitectura del Modelo Base:


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Capa_Entrada (Flatten)          │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Capa_Oculta_1 (Dense)           │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Capa_Oculta_2 (Dense)           │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Capa_Salida (Dense)             │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 109,386 (427.29 KB)

 Trainable params: 109,386 (427.29 KB)

 Non-trainable params: 0 (0.00 B)

## 4. Entrenamiento y Configuración de Hiperparámetros

En esta sección, realizaremos el entrenamiento de nuestra red neuronal. estableceremos primero un **modelo base** (Baseline) con hiperparámetros estándar. A partir de este modelo, realizaremos experimentos controlados variando un parámetro a la vez para analizar su impacto en la convergencia y precisión de la red.

**Configuración del Modelo Base:**
* **Épocas (Epochs):** 15. Un número moderado para observar la tendencia de aprendizaje sin incurrir en tiempos de ejecución excesivos en esta primera fase.
* **Tamaño de Batch (Batch Size):** 32. Es el estándar en la industria; ofrece un buen equilibrio entre la velocidad de convergencia y la estabilidad del gradiente.
* **Tasa de Aprendizaje (Learning Rate):** Utilizaremos la tasa por defecto del optimizador Adam (0.001), la cual ajustaremos dinámicamente en experimentos posteriores.
* **Validación:** Separaremos un 20% de los datos de entrenamiento (`validation_split=0.2`) para monitorear el desempeño en datos no vistos durante cada época y detectar posibles indicios de *overfitting*.

In [ ]:
# 5. Entrenamiento del Modelo Base
print("Iniciando entrenamiento del Modelo Base...")
historial_base = modelo_base.fit(
    train_images,
    train_labels,
    epochs=15,
    batch_size=32,
    validation_split=0.2 # 20% de los datos para validación
)

# 6. Visualización de Resultados (Curvas de Aprendizaje)
def graficar_historial(historial, titulo="Curvas de Entrenamiento"):
    plt.figure(figsize=(12, 4))

    # Gráfico de Precisión (Accuracy)
    plt.subplot(1, 2, 1)
    plt.plot(historial.history['accuracy'], label='Precisión Entrenamiento', marker='o')
    plt.plot(historial.history['val_accuracy'], label='Precisión Validación', marker='x')
    plt.title(f'{titulo} - Precisión')
    plt.xlabel('Época')
    plt.ylabel('Precisión')
    plt.legend()
    plt.grid(True)

    # Gráfico de Pérdida (Loss)
    plt.subplot(1, 2, 2)
    plt.plot(historial.history['loss'], label='Pérdida Entrenamiento', marker='o')
    plt.plot(historial.history['val_loss'], label='Pérdida Validación', marker='x')
    plt.title(f'{titulo} - Pérdida')
    plt.xlabel('Época')
    plt.ylabel('Pérdida')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

# Llamamos a la función para graficar el modelo base
graficar_historial(historial_base, "Modelo Base (Batch 32, Adam default)")

Iniciando entrenamiento del Modelo Base...
Epoch 1/15
 742/1500 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.7421 - loss: 0.7658 

Bueno en este modelo base podemos ver que al separar la fraccion en un 20% para la validacion vemos que tenemos sobreajuste de la epoca 6 en adelante. Con los datos de entrenamiento vemos que cada vez el modelo predice mejor y que tambien pasa lo mismo con la perdida, por el lado de nuestros set de validación ocurre que en el grafico de presicion se estanca ya que llego a su limite para generalizar y por eso no sube más, por el lado de la perdida es lo mismo llega a la epoca 6 y vuelve a subir y bajar sin mejoria de perdida.

### 4.1 Experimento Controlado: Impacto de la Tasa de Aprendizaje

Para analizar el efecto de la tasa de aprendizaje en la convergencia del modelo, realizaremos un experimento controlado manteniendo fija la arquitectura base del MLP, el tamaño de batch (32), la cantidad de épocas (15) y la proporción de validación (20%).

Se compararán tres configuraciones del optimizador Adam:

- Learning Rate = 0.0005
- Learning Rate = 0.001
- Learning Rate = 0.005

De esta forma, podremos observar cómo una tasa de aprendizaje baja, media o alta afecta la estabilidad del entrenamiento, la velocidad de convergencia y el desempeño final del modelo.

In [ ]:
# Experimento controlado: impacto del learning rate
from tensorflow.keras import layers, Sequential
from tensorflow.keras.optimizers import Adam
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def crear_modelo_mlp():
    modelo = Sequential([
        keras.Input(shape=(28, 28), name="Capa_Entrada"),
        layers.Flatten(),
        layers.Dense(128, activation='relu', name="Capa_Oculta_1"),
        layers.Dense(64, activation='relu', name="Capa_Oculta_2"),
        layers.Dense(10, activation='softmax', name="Capa_Salida")
    ])
    return modelo

learning_rates = [0.0005, 0.001, 0.005]
historiales_lr = {}
resultados_lr = []

for lr in learning_rates:
    print(f"\nEntrenando modelo con learning rate = {lr}")

    modelo_lr = crear_modelo_mlp()
    modelo_lr.compile(
        optimizer=Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    historial = modelo_lr.fit(
        train_images,
        train_labels,
        epochs=15,
        batch_size=32,
        validation_split=0.2,
        verbose=1
    )

    historiales_lr[lr] = historial

    predicciones = np.argmax(modelo_lr.predict(test_images, verbose=0), axis=1)
    acc = accuracy_score(test_labels, predicciones)
    prec, rec, f1, _ = precision_recall_fscore_support(
        test_labels,
        predicciones,
        average='macro'
    )

    resultados_lr.append({
        'Learning Rate': lr,
        'Accuracy': round(acc, 4),
        'Precision (Macro)': round(prec, 4),
        'Recall (Macro)': round(rec, 4),
        'F1-Score (Macro)': round(f1, 4)
    })

tabla_lr = pd.DataFrame(resultados_lr)
print("\n--- TABLA COMPARATIVA: IMPACTO DEL LEARNING RATE ---")
display(tabla_lr)

In [ ]:
# Comparación gráfica de accuracy y loss para distintos learning rates
plt.figure(figsize=(14, 5))

# Accuracy de validación
plt.subplot(1, 2, 1)
for lr, historial in historiales_lr.items():
    plt.plot(historial.history['val_accuracy'], marker='o', label=f'LR={lr}')
plt.title('Comparación de Precisión de Validación')
plt.xlabel('Época')
plt.ylabel('Val Accuracy')
plt.grid(True)
plt.legend()

# Loss de validación
plt.subplot(1, 2, 2)
for lr, historial in historiales_lr.items():
    plt.plot(historial.history['val_loss'], marker='x', label=f'LR={lr}')
plt.title('Comparación de Pérdida de Validación')
plt.xlabel('Época')
plt.ylabel('Val Loss')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

**Interpretación:**

- Tasa muy baja (0.0005):

  - Produce una convergencia ligeramente más estable y logra el mejor accuracy y F1 en test, pero la mejora frente a 0.001 es casi nada

- Tasa media (0.001):

  - Equilibra velocidad y estabilidad; es una buena configuración por defecto, muy cercana al rendimiento óptimo.

- Tasa alta (0.005):

  - Hace que el modelo aprenda de forma más agresiva, pero la función de pérdida de validación oscila más y el desempeño final cae, evidenciando que la red no converge a un mínimo estable.

En este problema concreto, valores en el rango [0.0005, 0.001] son adecuados para Adam; tasas mayores como 0.005 deterioran la capacidad de generalización del modelo.

### 4.2 Comparación de funciones de activación en el MLP

Además de ajustar hiperparámetros como la tasa de aprendizaje, es importante analizar cómo las funciones de activación influyen en el desempeño del modelo. Para ello, se realiza un experimento controlado comparando dos configuraciones del MLP:

- **Modelo ReLU**: capas ocultas con función de activación ReLU.
- **Modelo tanh**: capas ocultas con función de activación tanh.

En ambos casos se mantiene fija la arquitectura (128 y 64 neuronas en las capas ocultas), la función de pérdida (`sparse_categorical_crossentropy`), el optimizador Adam, la cantidad de épocas (15), el tamaño del batch (32) y el conjunto de entrenamiento/validación. De esta forma, cualquier diferencia en el desempeño final se puede atribuir principalmente a la función de activación utilizada.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers, Sequential
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def crear_modelo_mlp_relu():
    modelo = Sequential([
        keras.Input(shape=(28, 28), name="Capa_Entrada"),
        layers.Flatten(),
        layers.Dense(128, activation='relu', name="Capa_Oculta_1"),
        layers.Dense(64, activation='relu', name="Capa_Oculta_2"),
        layers.Dense(10, activation='softmax', name="Capa_Salida")
    ])
    return modelo

def crear_modelo_mlp_tanh():
    modelo = Sequential([
        keras.Input(shape=(28, 28), name="Capa_Entrada"),
        layers.Flatten(),
        layers.Dense(128, activation='tanh', name="Capa_Oculta_1"),
        layers.Dense(64, activation='tanh', name="Capa_Oculta_2"),
        layers.Dense(10, activation='softmax', name="Capa_Salida")
    ])
    return modelo

modelos_activacion = {
    'ReLU': crear_modelo_mlp_relu,
    'tanh': crear_modelo_mlp_tanh
}

historiales_act = {}
resultados_act = []

for nombre_act, fn_modelo in modelos_activacion.items():
    print(f"\nEntrenando modelo con activación {nombre_act}")

    modelo = fn_modelo()
    modelo.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    historial = modelo.fit(
        train_images,
        train_labels,
        epochs=15,
        batch_size=32,
        validation_split=0.2,
        verbose=1
    )

    historiales_act[nombre_act] = historial

    predicciones = np.argmax(modelo.predict(test_images, verbose=0), axis=1)
    acc = accuracy_score(test_labels, predicciones)
    prec, rec, f1, _ = precision_recall_fscore_support(
        test_labels,
        predicciones,
        average='macro'
    )

    resultados_act.append({
        'Función de activación': nombre_act,
        'Accuracy': round(acc, 4),
        'Precision (Macro)': round(prec, 4),
        'Recall (Macro)': round(rec, 4),
        'F1-Score (Macro)': round(f1, 4)
    })

tabla_act = pd.DataFrame(resultados_act)
print("\n--- TABLA COMPARATIVA: ACTIVACIÓN ReLU vs tanh ---")
display(tabla_act)

In [ ]:
# Comparación de accuracy y loss de validación para ReLU vs tanh
plt.figure(figsize=(14, 5))

# Accuracy de validación
plt.subplot(1, 2, 1)
for nombre_act, historial in historiales_act.items():
    plt.plot(historial.history['val_accuracy'], marker='o', label=nombre_act)
plt.title('Precisión de Validación: ReLU vs tanh')
plt.xlabel('Época')
plt.ylabel('Val Accuracy')
plt.grid(True)
plt.legend()

# Loss de validación
plt.subplot(1, 2, 2)
for nombre_act, historial in historiales_act.items():
    plt.plot(historial.history['val_loss'], marker='x', label=nombre_act)
plt.title('Pérdida de Validación: ReLU vs tanh')
plt.xlabel('Época')
plt.ylabel('Val Loss')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

Interpretación directa:

con estos graficos podemos comparar que la diferencia de rendimiento entre el accuracy y f1-score entre relu y tanh es minima de un 0.04% donde si es mayor relu. pero la desicion que tomamos es que tanh demuestra una convergencia mucho mas suave y estable donde no tiene estos saltos o picos como lo tiene ReLu obteniendo así tanh una mayor estabilidad y menor riesgo de oscilaciones en el descenso del gradiente

### 4.3 Ajuste del nivel de Dropout en el MLP

En el modelo optimizado se incorporó Dropout con una tasa de 0.3 como técnica de regularización para reducir el sobreajuste. Sin embargo, una tasa de Dropout demasiado alta puede eliminar demasiada información durante el entrenamiento y perjudicar el desempeño final del modelo.

Para analizar este efecto, se realiza un experimento controlado manteniendo fija la arquitectura base del MLP (128 y 64 neuronas), la función de activación tanh, el optimizador Adam, la cantidad de épocas (15), el tamaño del batch (32) y el conjunto de datos. Solo se varía la tasa de Dropout en las capas ocultas:

- Dropout = 0.1  
- Dropout = 0.2  
- Dropout = 0.3  

El objetivo es identificar qué nivel de Dropout ofrece un mejor equilibrio entre regularización (reducción del sobreajuste) y desempeño en el conjunto de prueba.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers, Sequential
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def crear_modelo_mlp_dropout(dropout_rate):
    modelo = Sequential([
        keras.Input(shape=(28, 28), name="Capa_Entrada"),
        layers.Flatten(),
        layers.Dense(128, activation='tanh', name="Capa_Oculta_1"),
        layers.Dropout(dropout_rate, name=f"Dropout_{dropout_rate}_1"),
        layers.Dense(64, activation='tanh', name="Capa_Oculta_2"),
        layers.Dropout(dropout_rate, name=f"Dropout_{dropout_rate}_2"),
        layers.Dense(10, activation='softmax', name="Capa_Salida")
    ])
    return modelo

dropout_rates = [0.1, 0.2, 0.3]
historiales_do = {}
resultados_do = []

for dr in dropout_rates:
    print(f"\nEntrenando modelo con Dropout = {dr}")

    modelo_do = crear_modelo_mlp_dropout(dr)
    modelo_do.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    historial = modelo_do.fit(
        train_images,
        train_labels,
        epochs=15,
        batch_size=32,
        validation_split=0.2,
        verbose=1
    )

    historiales_do[dr] = historial

    predicciones = np.argmax(modelo_do.predict(test_images, verbose=0), axis=1)
    acc = accuracy_score(test_labels, predicciones)
    prec, rec, f1, _ = precision_recall_fscore_support(
        test_labels,
        predicciones,
        average='macro'
    )

    resultados_do.append({
        'Dropout': dr,
        'Accuracy': round(acc, 4),
        'Precision (Macro)': round(prec, 4),
        'Recall (Macro)': round(rec, 4),
        'F1-Score (Macro)': round(f1, 4)
    })

tabla_do = pd.DataFrame(resultados_do)
print("\n--- TABLA COMPARATIVA: NIVELES DE DROPOUT ---")
display(tabla_do)

In [ ]:
# Comparación de accuracy y loss de validación para distintos niveles de Dropout
plt.figure(figsize=(14, 5))

# Accuracy de validación
plt.subplot(1, 2, 1)
for dr, historial in historiales_do.items():
    plt.plot(historial.history['val_accuracy'], marker='o', label=f'Dropout={dr}')
plt.title('Precisión de Validación para distintos niveles de Dropout')
plt.xlabel('Época')
plt.ylabel('Val Accuracy')
plt.grid(True)
plt.legend()

# Loss de validación
plt.subplot(1, 2, 2)
for dr, historial in historiales_do.items():
    plt.plot(historial.history['val_loss'], marker='x', label=f'Dropout={dr}')
plt.title('Pérdida de Validación para distintos niveles de Dropout')
plt.xlabel('Época')
plt.ylabel('Val Loss')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

Dropout moderado (0.1): Demuestra ser la tasa óptima, logrando el mejor equilibrio. Consigue la menor pérdida de validación y la mayor precisión (Accuracy: 88.19%, F1-Score: 88.12%). Al apagar solo el 10% de las neuronas, la red regulariza el sobreajuste exitosamente sin perder capacidad de retención de patrones.

Dropout alto (0.2 y 0.3): Resultan demasiado agresivos para esta arquitectura. La precisión decae al ~86.9% y la función de pérdida de validación empeora. Al combinar la compresión natural de la función tanh con un dropout elevado, la red sufre pérdida de información crítica por época, degradando su desempeño global.

Conclusión: El modelo con Dropout 0.1 es el ganador absoluto. No solo logra mitigar el overfitting, sino que mantiene un desempeño numérico a la par (e incluso marginalmente superior) al modelo base. Esta será la configuración de hiperparámetros elegida para la evaluación final del modelo.

## 5. Síntesis de Resultados y Construcción del Modelo Optimizado

Análisis de los Resultados (Diagnóstico): Al observar las curvas de entrenamiento del modelo base, detectamos un claro comportamiento de sobreajuste (overfitting) a partir de la época 6.

Justificación: La métrica de pérdida (loss) en el conjunto de entrenamiento continúa descendiendo hasta 0.20, pero la pérdida en el conjunto de validación deja de mejorar y comienza a oscilar al alza (rebotando cerca de 0.35). Esto indica que la red está memorizando los datos de entrenamiento en lugar de aprender patrones generalizables.

Ajuste Propuesto (Modelo Optimizado Definitivo): Para mitigar este problema y basándonos en la evidencia de nuestros experimentos controlados (Secciones 4.2 y 4.3), construiremos nuestro modelo final integrando las configuraciones óptimas descubiertas.

Justificación técnica de los ajustes: >   1. Activación Tanh: Reemplazaremos ReLU por Tanh en las capas ocultas, ya que demostró una convergencia mucho más estable y menor riesgo de oscilaciones bruscas.
2. Dropout (0.1): Agregaremos capas Dropout(0.1) después de cada capa densa oculta. Esto desactivará aleatoriamente solo el 10% de las neuronas durante cada paso de actualización. Al hacer esto, evitamos la coadaptación de las neuronas sin generar una pérdida excesiva de información (evitando el underfitting que causaba el 0.3), logrando el balance perfecto de regularización para este problema.

In [ ]:
# 7. Construcción del Modelo Optimizado Definitivo
modelo_optimizado = keras.Sequential([
    keras.Input(shape=(28, 28), name="Capa_Entrada_Input"), # Previene el UserWarning
    keras.layers.Flatten(name="Capa_Entrada"),

    keras.layers.Dense(128, activation='tanh', name="Capa_Oculta_1"), # <-- Cambiado a tanh
    keras.layers.Dropout(0.1, name="Dropout_1"), # <-- Apagamos el 10% de las neuronas

    keras.layers.Dense(64, activation='tanh', name="Capa_Oculta_2"),  # <-- Cambiado a tanh
    keras.layers.Dropout(0.1, name="Dropout_2"), # <-- Apagamos el 10% de las neuronas

    keras.layers.Dense(10, activation='softmax', name="Capa_Salida")
])

# 8. Compilación
modelo_optimizado.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# 9. Entrenamiento del Modelo Optimizado
print("Iniciando entrenamiento del Modelo Optimizado Definitivo...")
historial_optimizado = modelo_optimizado.fit(
    train_images,
    train_labels,
    epochs=15,
    batch_size=32,
    validation_split=0.2
)

# 10. Graficar los nuevos resultados
graficar_historial(historial_optimizado, "Modelo Optimizado (Tanh + Dropout 0.1)")

Conclusión del Modelo Optimizado (Configuración Final):
Al analizar las curvas de aprendizaje del modelo optimizado utilizando la función de activación Tanh y una tasa de Dropout de 0.1, podemos concluir lo siguiente: A diferencia del modelo base, la función de pérdida de validación (línea naranja) desciende de manera constante y se estabiliza asintóticamente sin mostrar rebotes significativos o tendencias al alza. Mitigación del Overfitting: Se logró reducir drásticamente la brecha entre el entrenamiento y la validación. Mientras que el modelo base presentaba una clara divergencia a partir de la época 6, este modelo optimizado mantiene ambas curvas cercanas, lo que demuestra una excelente capacidad de generalización. Punto de Equilibrio Óptimo: El uso de Dropout al 10% permitió regularizar la red sin incurrir en underfitting. Esto evitó la pérdida excesiva de información que observamos con tasas más altas (como 0.3), permitiendo que el modelo alcanzara una precisión superior al 88% en el conjunto de validación.

#### Comparación: Modelo Base vs Mejor Modelo con Dropout (0.1)

A partir del experimento de ajuste de Dropout realizado en la sección 4.3, se identificó que Dropout = 0.1 ofrece el mejor equilibrio entre regularización y desempeño. A continuación se comparan ambos modelos en el conjunto de prueba:

In [ ]:
import pandas as pd

import pandas as pd

# Extraemos los datos de las tablas previas para asegurar consistencia total
# tabla_lr.iloc[1] contiene los resultados del modelo base (LR=0.001)
# tabla_do.iloc[0] contiene los resultados del mejor Dropout (0.1)

tabla_base_vs_do1 = pd.DataFrame([
    {
        'Modelo': 'Base (Original - ReLU)',
        'Accuracy': tabla_lr.iloc[1]['Accuracy'],
        'Precision (Macro)': tabla_lr.iloc[1]['Precision (Macro)'],
        'Recall (Macro)': tabla_lr.iloc[1]['Recall (Macro)'],
        'F1-Score (Macro)': tabla_lr.iloc[1]['F1-Score (Macro)']
    },
    {
        'Modelo': 'Optimizado (Final - Tanh + Dropout 0.1)',
        'Accuracy': tabla_do.iloc[0]['Accuracy'],
        'Precision (Macro)': tabla_do.iloc[0]['Precision (Macro)'],
        'Recall (Macro)': tabla_do.iloc[0]['Recall (Macro)'],
        'F1-Score (Macro)': tabla_do.iloc[0]['F1-Score (Macro)']
    }
])

print("--- COMPARACIÓN FINAL: MODELO BASE VS MODELO OPTIMIZADO ---")
display(tabla_base_vs_do1)

nota: Aqui en las metricas podemos ver que el modelo base muestra ser superior ligeramente pero hay que tener en cuenta que esas metricas no necesariamente indican que el modelo predice de buena manera, el accuracy por si solo es insuficiente para verificar el comportamiento de un modelo. No refleja la capacidad de generalización del modelo. por eso es mejor verlo graficado de tal manera que podemos ver el descenso del gradiente a traves de las curvas de aprendizaje.

## 6. Evaluación Final y Comparación de Modelos

In [ ]:
import pandas as pd

# Creamos la tabla final extrayendo TODO directamente de las tablas de experimentos previos
# Esto asegura que los datos sean 100% consistentes con tus gráficos y ejecuciones.

tabla_final = pd.DataFrame([
    {
        'Experimento': 'Modelo Base (Baseline)',
        'Configuración': 'ReLU, LR=0.001, sin Dropout',
        'Accuracy': tabla_lr.iloc[1]['Accuracy'],
        'F1-Score': tabla_lr.iloc[1]['F1-Score (Macro)']
    },
    {
        'Experimento': 'Tasa de Aprendizaje Baja',
        'Configuración': 'LR=0.0005',
        'Accuracy': tabla_lr.iloc[0]['Accuracy'],
        'F1-Score': tabla_lr.iloc[0]['F1-Score (Macro)']
    },
    {
        'Experimento': 'Tasa de Aprendizaje Alta',
        'Configuración': 'LR=0.005',
        'Accuracy': tabla_lr.iloc[2]['Accuracy'],
        'F1-Score': tabla_lr.iloc[2]['F1-Score (Macro)']
    },
    {
        'Experimento': 'Función de Activación Tanh',
        'Configuración': 'tanh en capas ocultas',
        'Accuracy': tabla_act.iloc[1]['Accuracy'],
        'F1-Score': tabla_act.iloc[1]['F1-Score (Macro)']
    },
    {
        'Experimento': 'Función de Activación ReLU',
        'Configuración': 'ReLU en capas ocultas',
        'Accuracy': tabla_act.iloc[0]['Accuracy'],
        'F1-Score': tabla_act.iloc[0]['F1-Score (Macro)']
    },
    {
        'Experimento': 'Optimización: Dropout 0.1',
        'Configuración': 'Regularización Suave (Ganador)',
        'Accuracy': tabla_do.iloc[0]['Accuracy'],
        'F1-Score': tabla_do.iloc[0]['F1-Score (Macro)']
    },
    {
        'Experimento': 'Optimización: Dropout 0.2',
        'Configuración': 'Regularización Media',
        'Accuracy': tabla_do.iloc[1]['Accuracy'],
        'F1-Score': tabla_do.iloc[1]['F1-Score (Macro)']
    },
    {
        'Experimento': 'Optimización: Dropout 0.3',
        'Configuración': 'Regularización Fuerte',
        'Accuracy': tabla_do.iloc[2]['Accuracy'],
        'F1-Score': tabla_do.iloc[2]['F1-Score (Macro)']
    },
])

print("=== RESUMEN FINAL Y SELECCIÓN DEL MODELO OPTIMIZADO ===")
display(tabla_final)

**Cálculo e Interpretación de Métricas:**
Para evaluar el desempeño real de nuestra red neuronal, utilizaremos el conjunto de datos de prueba (`test_images`), el cual la red no ha visto durante el proceso de entrenamiento. Evaluar sobre datos no vistos es la única forma de medir la capacidad de generalización del modelo.

Calcularemos las siguientes métricas clave utilizando la librería `scikit-learn`:
* **Accuracy (Exactitud):** Porcentaje total de imágenes clasificadas correctamente.
* **Precision (Precisión):** De todas las prendas que el modelo clasificó como una categoría específica (ej. 'Pantalón'), cuántas realmente lo eran.
* **Recall (Sensibilidad):** De todas las prendas que *realmente* eran de una categoría, cuántas logró identificar correctamente el modelo.
* **F1-Score:** La media armónica entre Precision y Recall. Es un indicador robusto del desempeño general por clase.

**Análisis Comparativo (Base vs. Optimizado):**
Al comparar las configuraciones, el objetivo es evidenciar cómo la regularización (Dropout) afecta el resultado final en datos de prueba. Si bien el modelo base puede tener una precisión más alta en el conjunto de entrenamiento (debido al *overfitting*), el modelo optimizado debería presentar una mayor estabilidad y, potencialmente, un mejor equilibrio entre *Precision* y *Recall* en el conjunto de pruebas, ya que aprendió características más generales y no "memorizó" el dataset.

In [ ]:
import pandas as pd
import numpy as np

# Predicciones del modelo base y optimizado (Dropout 0.3)
predicciones_base = np.argmax(modelo_base.predict(test_images, verbose=0), axis=1)
predicciones_opt  = np.argmax(modelo_optimizado.predict(test_images, verbose=0), axis=1)

def obtener_metricas(y_true, y_pred):
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro')
    return [round(acc,4), round(prec,4), round(rec,4), round(f1,4)]

metricas_base   = obtener_metricas(test_labels, predicciones_base)
metricas_opt    = obtener_metricas(test_labels, predicciones_opt)
metricas_do1    = [
    tabla_do.iloc[0]['Accuracy'],
    tabla_do.iloc[0]['Precision (Macro)'],
    tabla_do.iloc[0]['Recall (Macro)'],
    tabla_do.iloc[0]['F1-Score (Macro)']
]

tabla_comparativa = pd.DataFrame({
    'Métrica'                  : ['Accuracy','Precision (Macro)','Recall (Macro)','F1-Score (Macro)'],
    'Modelo Base'              : metricas_base,
    'Dropout 0.1 (Mejor)'     : metricas_do1,
    'Dropout 0.3 (Original)'  : metricas_opt
})

print("\n--- TABLA COMPARATIVA DE DESEMPEÑO ---")
display(tabla_comparativa)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np

predicciones_base = np.argmax(modelo_base.predict(test_images, verbose=0), axis=1)

class_names = ['Camiseta', 'Pantalón', 'Suéter', 'Vestido', 'Abrigo',
               'Sandalia', 'Camisa', 'Zapatilla', 'Bolso', 'Botín']

cm = confusion_matrix(test_labels, predicciones_base)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names)
plt.title('Matriz de Confusión - Modelo Base')
plt.ylabel('Etiqueta Real')
plt.xlabel('Etiqueta Predicha')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

**Interpretación de la Matriz de Confusión:**

La matriz confirma visualmente lo analizado en las métricas por clase:

- Las clases con **siluetas únicas** como Pantalón, Bolso, Sandalia y Botín concentran casi todos sus valores en la diagonal, lo que indica clasificaciones casi perfectas.
- La **Camisa** es la clase más problemática: se confunde frecuentemente con Camiseta y Suéter, ya que comparten formas similares en imágenes de 28×28 píxeles en escala de grises.
- El **Suéter** también presenta confusión con Abrigo y Camiseta por la misma razón: prendas de ropa superior con siluetas parecidas.
- Esto ratifica que la limitación principal del MLP es el uso de `Flatten`, que elimina la información espacial de la imagen y dificulta distinguir patrones locales entre clases similares.

In [ ]:
# Generar predicciones con el modelo optimizado
predicciones_opt = np.argmax(modelo_optimizado.predict(test_images, verbose=0), axis=1)

# Crear la matriz de confusión
cm_opt = confusion_matrix(test_labels, predicciones_opt)

# Graficar
plt.figure(figsize=(10, 8))
sns.heatmap(cm_opt, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicción del Modelo')
plt.ylabel('Clase Real')
plt.title('Matriz de Confusión: Modelo Optimizado (Tanh + Dropout 0.1)')
plt.show()

El análisis de las matrices de confusión revela que los errores del modelo no son aleatorios, sino que se concentran en clases con alta similitud morfológica (Camisa, Camiseta y Suéter). Mientras que el modelo base intenta 'forzar' clasificaciones que llevan al overfitting, el modelo optimizado presenta una diagonal principal sólida y una distribución de errores más coherente, demostrando que la arquitectura con Tanh y Dropout 0.1 ha aprendido características generales más estables.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Mostrar 5 imágenes del test con su predicción real vs predicha
class_names = ['Camiseta','Pantalón','Suéter','Vestido','Abrigo',
               'Sandalia','Camisa','Zapatilla','Bolso','Botín']

predicciones = np.argmax(modelo_base.predict(test_images[:10], verbose=0), axis=1)

plt.figure(figsize=(15, 3))
for i in range(5):
    plt.subplot(1, 5, i+1)
    plt.imshow(test_images[i], cmap='gray')
    color = 'green' if predicciones[i] == test_labels[i] else 'red'
    plt.title(f"Real: {class_names[test_labels[i]]}\nPred: {class_names[predicciones[i]]}",
              color=color, fontsize=9)
    plt.axis('off')
plt.suptitle('Predicciones del Modelo Base (verde=correcto, rojo=error)')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Definimos el rango (del 1 al 15)
inicio = 1
fin = 16 # El límite superior no se incluye, por eso usamos 16
num_imagenes = fin - inicio

# 2. Generar predicciones para ese rango específico con el modelo optimizado
# Usamos el slice [inicio:fin] para tomar exactamente esas imágenes [cite: 1146]
predicciones_opt = np.argmax(modelo_optimizado.predict(test_images[inicio:fin], verbose=0), axis=1)

# 3. Configurar la cuadrícula (3 filas y 5 columnas para las 15 imágenes)
filas = 3
columnas = 5
plt.figure(figsize=(15, 10)) # Aumentamos el alto (10) para las 3 filas

for i in range(num_imagenes):
    plt.subplot(filas, columnas, i + 1)

    # Calculamos el índice real en el dataset original
    indice_real = inicio + i
    plt.imshow(test_images[indice_real], cmap='gray')

    # Comparamos la predicción con la etiqueta real [cite: 1160]
    es_correcto = predicciones_opt[i] == test_labels[indice_real]
    color = 'green' if es_correcto else 'red'

    # Mostramos los nombres de las clases [cite: 1204, 1205]
    plt.title(f"ID: {indice_real}\nReal: {class_names[test_labels[indice_real]]}\nPred: {class_names[predicciones_opt[i]]}",
              color=color, fontsize=9)
    plt.axis('off')

plt.suptitle(f'Resultados del Modelo Optimizado: Imágenes {inicio} a {fin-1}', fontsize=16)
plt.tight_layout()
plt.subplots_adjust(top=0.92) # Espacio para el título principal
plt.show()

La visualización de las predicciones confirma empíricamente los resultados obtenidos en las tablas de métricas. El modelo optimizado no solo presenta una brecha de generalización mínima en las curvas de aprendizaje, sino que demuestra una alta fiabilidad al clasificar correctamente imágenes con diferentes densidades de píxeles y formas, validando la eficacia de las técnicas de regularización aplicadas.

## 7. Conclusiones y Posibles Mejoras


Tras realizar una experimentación sistemática ajustando la tasa de aprendizaje, funciones de activación y técnicas de regularización, se determinó que la configuración con función de activación Tanh y Dropout de 0.1 es la más adecuada para este problema.

Aunque el modelo base obtuvo un Accuracy marginalmente superior (88.73%), el análisis de las curvas de aprendizaje reveló una divergencia significativa a partir de la época 6, indicando un claro fenómeno de overfitting (memorización). En contraste, el modelo con Dropout logró una convergencia estable y una generalización mucho más confiable, sacrificando apenas un 0.84% de precisión bruta a cambio de una robustez superior frente a datos no vistos.

Diagnóstico por Categorías (El desafío morfológico)
El análisis detallado mediante matrices de confusión y reportes de clasificación muestra que el modelo es extremadamente eficaz en prendas con siluetas geométricas únicas (Pantalón F1: 0.98, Bolso F1: 0.98). Sin embargo, existe un "triángulo de confusión" persistente entre Camisa, Camiseta y Suéter.

La Camisa presenta el desempeño más bajo (F1: 0.67), evidenciando que la red tiene dificultades para distinguir patrones locales sutiles, como cuellos o botones, en una resolución de 28x28 píxeles.

Propuestas de Mejora y Visión a Futuro
La limitación principal radica en la arquitectura del Perceptrón Multicapa (MLP). El uso de la capa Flatten convierte la imagen en un vector unidimensional, lo que provoca la pérdida de la relación espacial entre los píxeles (la estructura de "vecindad" que define una prenda). Para superar esto y mejorar el desempeño, se propone:

Redes Neuronales Convolucionales (CNN): Implementar capas de convolución que, mediante filtros, permitan extraer características espaciales y patrones locales sin destruir la estructura de la imagen.

Early Stopping: Integrar un mecanismo de parada temprana para detener el entrenamiento en el punto exacto de mínima pérdida de validación, automatizando la búsqueda de la mejor época.

Aumento de Datos (Data Augmentation): Aplicar rotaciones o desplazamientos leves a las imágenes de entrenamiento para que la red aprenda a reconocer las prendas sin importar su posición exacta, aumentando la robustez del sistema.